# Python starter notebook

**Campus Survival Pack — ictcampus.lk**

Load data, clean it, group it, chart it. Five steps, about twenty minutes.

> **The data in here is synthetic.** It is shaped like Sri Lankan district
> statistics so the exercises feel real, but the numbers were generated.
> Never cite them. For real figures use the Department of Census and
> Statistics, or the Central Bank of Sri Lanka's *Annual Economic Review*,
> and cite the publication itself.

## How to run this on a phone

1. Go to **colab.research.google.com** and sign in with a Google account.
2. **File → Upload notebook**, and pick this `.ipynb` file.
3. Upload `sri-lanka-districts-synthetic.csv` too: the folder icon on the left, then the upload button.
4. Run each cell with the ▶ button, top to bottom.

Nothing to install. Colab already has pandas and matplotlib.


## 1. Load it

`pd.read_csv` reads the file into a **DataFrame** — a table with named columns.
`comment='#'` tells it to ignore the warning line at the top of the file.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("sri-lanka-districts-synthetic.csv", comment="#")

print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()


`.head()` shows the first five rows. Always look at your data before you do
anything to it — half the mistakes in a student project are a column that is
not what its name suggests.


In [ ]:
# What type is each column, and how many values are missing?
df.info()


## 2. Clean it

This file has two problems on purpose: one missing value and one duplicated row.
Real data always has something. Find it before you analyse it, not after.


In [ ]:
print("Missing values per column:")
print(df.isna().sum())

print(f"\nDuplicate rows: {df.duplicated().sum()}")


In [ ]:
# Drop the duplicate, keeping the first copy.
df = df.drop_duplicates()

# Fill the missing percentage with that district's own average rather than
# the national one — a missing Jaffna value is better guessed from Jaffna.
df["internet_users_pct"] = df.groupby("district")["internet_users_pct"].transform(
    lambda s: s.fillna(s.mean())
)

print(f"{len(df)} rows after cleaning")
print(f"Missing values left: {df.isna().sum().sum()}")


**Say what you did.** In your assignment, write down that you removed a
duplicate and how you filled the gap. A marker cannot reproduce your numbers
otherwise, and 'I cleaned the data' is not a method.


## 3. Ask it something

`groupby` splits the table into groups, applies a calculation to each, and puts
the answers back together. It is the single most useful thing pandas does.


In [ ]:
# Average income per province, most recent year, largest first.
latest = df[df["year"] == df["year"].max()]

by_province = (
    latest.groupby("province")["avg_monthly_income_lkr"]
    .mean()
    .sort_values(ascending=False)
    .round(0)
)

by_province


In [ ]:
# Several numbers at once.
latest.groupby("province").agg(
    districts=("district", "count"),
    population=("population", "sum"),
    mean_internet_pct=("internet_users_pct", "mean"),
).round(1)


## 4. Chart it

One chart, one point. A chart that shows everything shows nothing.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

by_province.plot(kind="barh", ax=ax, color="#E8722C")

ax.set_xlabel("Average monthly income (Rs)")
ax.set_ylabel("")
ax.set_title(f"Average monthly income by province, {df['year'].max()} (synthetic data)")
ax.invert_yaxis()
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
# A trend over time: mean internet users per year, all districts.
trend = df.groupby("year")["internet_users_pct"].mean()

fig, ax = plt.subplots(figsize=(7, 4))
trend.plot(marker="o", ax=ax, color="#E8722C")

ax.set_ylabel("Internet users (%)")
ax.set_xlabel("Year")
ax.set_title("Mean internet use by year (synthetic data)")
ax.set_xticks(sorted(df["year"].unique()))
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()


**Every chart needs a caption saying what it shows and where the data came
from.** If the data is synthetic, the caption says so — as these do.


## 5. Get it out

Save the figure at 200 dpi or better, or it will look blurry in your document.


In [ ]:
fig.savefig("internet-trend.png", dpi=200, bbox_inches="tight")

# And the grouped table, if you want it in Excel.
by_province.to_csv("income-by-province.csv")

print("Saved. In Colab, open the folder icon on the left to download them.")


## Now change something

1. Chart population instead of income.
2. Group by `district` rather than `province` and show only the top ten.
3. Work out the percentage change in income from 2021 to 2025, per province.
4. Load your own CSV. Everything above works unchanged if your columns are tidy —
   one row per observation, one column per variable.

---

Campus Survival Pack — ictcampus.lk. Drafted with AI and reviewed by
Dr. Yasas Sri Wickramasinghe. Not accredited by any university.
